In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install dagshub mlflow
import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment2', mlflow=True)

import mlflow
mlflow.set_experiment("RandomForest_Training")
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as smama23

Initialized MLflow to track repo "smama23/MLassignment2"

Repository smama23/MLassignment2 initialized!

🏃 View run funny-stag-122 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/f36c9fdb0985446797ad33060ed01c4d
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [4]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

Cleaning

In [5]:
with mlflow.start_run(run_name="RandomForest_Cleaning_v1"):
    
    df_clean = df.copy()
    
    y = df_clean["isFraud"]
    df_clean = df_clean.drop(columns=["isFraud"])
    
    df_clean = df_clean.fillna(-999)
    
    mlflow.log_param("cleaning", "fillna_-999")
    mlflow.log_metric("num_features", df_clean.shape[1])

🏃 View run RandomForest_Cleaning_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/cdd99f6213f34764b2da415a5f96c0ed
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


Feature Engineering

In [6]:
with mlflow.start_run(run_name="RandomForest_FeatureEngineering_v1"):
    
    df_fe = df_clean.copy()
    
    df_fe["hour"] = (df_fe["TransactionDT"] // 3600) % 24
    df_fe["day"] = (df_fe["TransactionDT"] // (3600 * 24)) % 7
    df_fe["log_amount"] = np.log1p(df_fe["TransactionAmt"])
    
    mlflow.log_param("features_added", "hour, day, log_amount")
    mlflow.log_metric("num_features_after_fe", df_fe.shape[1])

🏃 View run RandomForest_FeatureEngineering_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/39d0f207efae4da98c7f88c13109427e
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [7]:
def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)
    return df

Feature Selection

In [8]:
with mlflow.start_run(run_name="RandomForest_FeatureSelection_v1"):
    
    df_enc = df_fe.copy()
    
    cat_cols = df_enc.select_dtypes(include="object").columns
    df_enc = frequency_encoding(df_enc, cat_cols)
    
    nunique = df_enc.nunique()
    df_fs = df_enc[nunique[nunique > 1].index]
    
    mlflow.log_param("encoding", "frequency")
    mlflow.log_param("feature_selection", "remove_constant")
    mlflow.log_metric("num_features_final", df_fs.shape[1])

🏃 View run RandomForest_FeatureSelection_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/db39b4e5a05e4f3992d288c043a00f45
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


Training

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
with mlflow.start_run(run_name="RandomForest_Training_v1"):
    
    X_train, X_val, y_train, y_val = train_test_split(
        df_fs, y, test_size=0.2, stratify=y, random_state=42
    )
    
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/03 18:59:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 18:59:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


1.0 0.9318320504340569
🏃 View run RandomForest_Training_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/627c035fc4e54ce59a0acfa3e20c5d41
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [11]:
with mlflow.start_run(run_name="RandomForest_Training_v2_tuned"):
    
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_split", 10)
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/03 19:02:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 19:02:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8875970624894408 0.8774075925891822
🏃 View run RandomForest_Training_v2_tuned at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/eaf449251cb64c158dd49e02669fa5a6
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [12]:
with mlflow.start_run(run_name="RandomForest_Training_v3_balanced"):
    
    model = RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=5,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_split", 5)
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/03 19:05:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 19:05:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.9461791738958089 0.911034421561178
🏃 View run RandomForest_Training_v3_balanced at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/82429bf5ba084b7cb2ac9d79a05a835b
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [13]:
import pandas as pd

feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
top_features = feature_importance.sort_values(ascending=False)

print(top_features)

V258    2.662454e-02
C1      2.371570e-02
V201    2.265222e-02
V200    2.036077e-02
C13     1.902150e-02
            ...     
V120    8.450353e-05
V117    8.434732e-05
C3      2.423934e-05
V107    1.360502e-05
V305    3.661090e-08
Length: 436, dtype: float64


In [14]:
top_50 = top_features.head(50)

In [15]:
mlflow.log_dict(top_50.to_dict(), "rf_top_50_features.json")

In [16]:
mlflow.end_run(status="FINISHED")

with mlflow.start_run(run_name="RandomForest_FeatureSelection_Top50"):
    
    selected_cols = top_50.index
    
    X_train_sel = X_train[selected_cols]
    X_val_sel = X_val[selected_cols]
    
    model = RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=5,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
    
    model.fit(X_train_sel, y_train)
    
    y_train_pred = model.predict_proba(X_train_sel)[:, 1]
    y_val_pred = model.predict_proba(X_val_sel)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("feature_selection", "top_50_importance")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    print(train_auc, val_auc)

🏃 View run adventurous-gnu-420 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/f855fbbb9fe644f19200e873cd9f779d
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2
0.9528618004405872 0.9125678228938252
🏃 View run RandomForest_FeatureSelection_Top50 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/931a36edeca244ccaf24c3f7f980f754
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2


In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            freq = X[col].value_counts() / len(X)
            self.freq_maps[col] = freq
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["hour"] = X["TransactionDT"]
        X["day"] = X["TransactionDT"]
        X["log_amount"] = np.log1p(X["TransactionAmt"])
        return X


class FillNaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=-999):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.fillna(self.fill_value)

In [18]:
cat_cols = df.drop(columns=["isFraud"]).select_dtypes(include="object").columns

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("cleaning", FillNaTransformer()),
    ("feature_engineering", FeatureEngineer()),
    ("encoding", FrequencyEncoder(cat_cols)),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=5,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ))
])

In [20]:
mlflow.end_run(status="FINISHED")

with mlflow.start_run(run_name="RandomForest_Pipeline_v1"):
    
    X = df.drop(columns=["isFraud"])
    y = df["isFraud"]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    pipeline.fit(X_train, y_train)
    
    y_train_pred = pipeline.predict_proba(X_train)[:, 1]
    y_val_pred = pipeline.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("model", "RandomForest_pipeline")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(pipeline, "rf_pipeline_model")
    
    print(train_auc, val_auc)

2026/05/03 19:11:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 19:11:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.9489912900853394 0.9128831498538903
🏃 View run RandomForest_Pipeline_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2/runs/47431f51a4c74725b11e34f64535f372
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/2
